# Notebook 02 — RibFrac 500-Patient Preprocessing: Resumable Kaggle Dataset Batches

**Revised from the working `02-preprocessing-500-patients (2).ipynb`.** This notebook replaces the overlapping smoke-test, checkpoint-merging and upload cells with one coherent workflow. It preserves the source cohort definitions, authoritative **350 training / 75 validation / 75 internal test** assignments, and the established CT preprocessing: **1-mm isotropic resampling, HU clipping [-1000, 400], normalization to [0, 1], linear CT interpolation, nearest-neighbor label interpolation**.

### Data and storage strategy

- The **49 verified CT–label pairs** already published at `waboke/ribfrac-preprocessed-final` are **read-only** and are **never copied into `/kaggle/working`**.
- The remaining **451 patients** have a fixed, deterministic **19-dataset plan**: validation (3 × 25), training (12 × 25 + 14), internal test (2 × 25 + 12). No patient assignments are regenerated.
- Each dataset holds **at most 25 new patient pairs**; process **up to 25 patients per processing execution, targeting one complete published dataset per Kaggle session**. Output for the current batch is staged under `/kaggle/working/ribfrac_batch_workspace/batches/<batch_name>`.
- Publish an entire completed batch using `kagglehub.dataset_upload`. Uploading **does not** directly write to `/kaggle/input`, which is read-only.
- **Publishing and deletion are disabled by default**. You must explicitly enable each step. Never delete staging data until the *newly published* dataset is reattached and verified.
- Scanning attached published batches is essential for accurate progress. Attach the 49-patient baseline and all earlier batch datasets to every new Kaggle session.
- The original `test` patients can be preprocessed and stored, but must **never** be used for training, loss selection, tuning, or pilot validation.

### Processing 25 patients in one Kaggle session

`PATIENTS_PER_RUN = 25` permits the processing cell to attempt all 25 patients in the selected batch in one execution. **A Kaggle session is not automatically a durable copy**: the staged files under `/kaggle/working` must be uploaded to a distinct Kaggle dataset and the uploaded copy reattached and verified **before ending the session or deleting temporary data**. The previous 49 published patients are not copied into working storage.

**Storage caveat:** The 49 previously preprocessed pairs average 0.367 GiB each (25 ≈ 9.18 GiB), but 25 pairs at the previously observed 95th-percentile size could occupy about **15.11 GiB**, leaving limited room for temporary files on a session with 19.50 GiB free. The notebook monitors free space and may **stop safely before reaching 25**. If it does, keep the session/staged files or publish a separately planned smaller batch; do not assume you can restart a new session and recover unpersisted working files.

### Execution order

Run **Cells 1–10** in order, inspect the preflight output, change `RUN_PROCESSING` to `True`, and run the processing cell to attempt up to 25 patients; inspect its result and only repeat if it stops early and sufficient disk space is available. When 25/25 pairs are verified, separately enable publishing. After attaching the published dataset as an input, run the verification and optional cleanup cells. Change `SELECTED_BATCH` for the next dataset and rerun setup cells.

> **Remote source dependency:** The original notebook downloads individual NIfTI files using HTTP range requests from Zenodo ZIP archives. This notebook preserves that strategy, adds local raw NIfTI support if such files are attached, and fails safely if Zenodo blocks range access. Internet access may be required on Kaggle.

## Cell 1 — Imports and environment

In [ ]:
import gc
import hashlib
import json
import math
import os
import re
import shutil
import struct
import tempfile
import time
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import requests
from scipy.ndimage import zoom

print("Notebook 02 dependencies loaded.")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Nibabel:", nib.__version__)

## Cell 2 — Paths, fixed split and explicit safety controls

In [ ]:
# Change ONLY SELECTED_BATCH when moving to the next published dataset.
BASELINE_ROOT = Path("/kaggle/input/datasets/waboke/ribfrac-preprocessed-final")
WORKSPACE = Path("/kaggle/working/ribfrac_batch_workspace")
SELECTED_BATCH = "ribfrac-preprocessed-validation-001"
DATASET_OWNER = "waboke"
BATCH_SIZE = 25
PATIENTS_PER_RUN = 25
MIN_FREE_GIB = 3.5
RETRY_FAILED = True

# Safety gates: intentional defaults do not start expensive operations.
RUN_PROCESSING = False
PUBLISH_BATCH = False
PUBLISH_CONFIRMATION = ""   # Set to exactly SELECTED_BATCH when publishing.
DELETE_STAGING_AFTER_VERIFICATION = False
DELETE_CONFIRMATION = ""    # Set to exactly SELECTED_BATCH to clean verified staging.

EXPECTED_SPLITS = {"train": 350, "validation": 75, "test": 75}
EXPECTED_BASELINE_PAIRS = 49
GiB = 1024 ** 3
PLAN_DIR = WORKSPACE / "plans"
BATCHES_DIR = WORKSPACE / "batches"
TEMP_DIR = WORKSPACE / "tmp"

assert BASELINE_ROOT.is_dir(), (
    "Attach dataset waboke/ribfrac-preprocessed-final as a Kaggle input."
)
assert BATCH_SIZE == 25 and 1 <= PATIENTS_PER_RUN <= BATCH_SIZE
assert SELECTED_BATCH.startswith("ribfrac-preprocessed-")
for folder in (PLAN_DIR, BATCHES_DIR, TEMP_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Baseline input:", BASELINE_ROOT)
print("Workspace:", WORKSPACE)
print("Selected dataset:", SELECTED_BATCH)
print("Patients per execution:", PATIENTS_PER_RUN)
print("Preprocessing enabled:", RUN_PROCESSING)
print("Publishing enabled:", PUBLISH_BATCH)
print("Deletion enabled:", DELETE_STAGING_AFTER_VERIFICATION)

## Cell 3 — Recover the authoritative 500-patient registry and original source definitions

In [ ]:
# Exact original Zenodo source cohort configuration retained.
SOURCES = {
    "train_part1": {
        "expected_patients": 300,
        # Retain the original notebook's train_part1 Zenodo HTTP 403 workaround.
        "image_url": "https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip?download=1",
        "label_url": "https://zenodo.org/records/3893508/files/ribfrac-train-labels-1.zip?download=1",
    },
    "train_part2": {
        "expected_patients": 120,
        "image_url": "https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip",
        "label_url": "https://zenodo.org/records/3893498/files/ribfrac-train-labels-2.zip",
    },
    "validation": {
        "expected_patients": 80,
        "image_url": "https://zenodo.org/records/3893496/files/ribfrac-val-images.zip",
        "label_url": "https://zenodo.org/records/3893496/files/ribfrac-val-labels.zip",
    },
}

SPLIT_PATH = BASELINE_ROOT / "metadata/patient_split_500.csv"
METADATA_PATH = BASELINE_ROOT / "metadata/combined_ribfrac_metadata.csv"
assert SPLIT_PATH.is_file(), f"Missing authoritative split: {SPLIT_PATH}"
assert METADATA_PATH.is_file(), f"Missing annotation registry: {METADATA_PATH}"

patient_split = pd.read_csv(SPLIT_PATH, dtype=str, keep_default_na=False)
patient_split = patient_split.rename(columns={"patient_id": "public_id"})
assert {"public_id", "split"}.issubset(patient_split.columns)
patient_split = patient_split[["public_id", "split"]].copy()
patient_split["public_id"] = patient_split["public_id"].str.strip()
patient_split["split"] = patient_split["split"].str.strip().str.lower().replace({"val": "validation"})
assert len(patient_split) == 500 and patient_split["public_id"].nunique() == 500
assert patient_split["split"].value_counts().to_dict() == EXPECTED_SPLITS

combined_metadata = pd.read_csv(METADATA_PATH, dtype=str, keep_default_na=False)
if "source_partition" in combined_metadata.columns and "source" not in combined_metadata.columns:
    combined_metadata = combined_metadata.rename(columns={"source_partition": "source"})
assert {"public_id", "source", "label_id", "label_code"}.issubset(combined_metadata.columns)
combined_metadata["public_id"] = combined_metadata["public_id"].str.strip()
combined_metadata["source"] = combined_metadata["source"].str.strip()
assert combined_metadata["public_id"].ne("").all()
assert set(combined_metadata["public_id"]) == set(patient_split["public_id"])

source_lookup = combined_metadata[["public_id", "source"]].drop_duplicates()
assert source_lookup["public_id"].nunique() == 500
assert not source_lookup["public_id"].duplicated().any()
assert source_lookup.groupby("source")["public_id"].nunique().to_dict() == {
    name: cfg["expected_patients"] for name, cfg in SOURCES.items()
}
patient_registry = patient_split.merge(source_lookup, on="public_id", validate="one_to_one")
assert len(patient_registry) == 500 and patient_registry["source"].notna().all()

# A fixed baseline is essential: changing the set changes the 19 batch names.
def complete_pair_ids(root):
    img = {p.name.removesuffix("-image.npy") for p in (root / "images").glob("*-image.npy")}
    lbl = {p.name.removesuffix("-label.npy") for p in (root / "labels").glob("*-label.npy")}
    if img != lbl:
        raise RuntimeError(f"Incomplete CT/label files in {root}: image-only={img-lbl}, label-only={lbl-img}")
    return img

baseline_ids = complete_pair_ids(BASELINE_ROOT)
assert len(baseline_ids) == EXPECTED_BASELINE_PAIRS, (
    "The baseline dataset has changed since the 49-patient plan. "
    "Review the plan before proceeding; never silently repartition."
)
assert baseline_ids.issubset(set(patient_registry["public_id"]))
for pid in sorted(baseline_ids):
    for kind in ("image", "label"):
        p = BASELINE_ROOT / ("images" if kind == "image" else "labels") / f"{pid}-{kind}.npy"
        a = np.load(p, mmap_mode="r", allow_pickle=False)
        assert a.ndim == 3
        assert a.dtype == (np.float32 if kind == "image" else np.int16)
        del a

print("Authoritative split:", patient_registry["split"].value_counts().to_dict())
print("Existing verified pairs:", len(baseline_ids))
print("Source cohorts:", source_lookup.groupby("source")["public_id"].nunique().to_dict())
print("Patients still to preprocess:", 500 - len(baseline_ids))

## Cell 4 — Build the **fixed** 19-batch plan (never renumber earlier batches)

In [ ]:
# The plan is always derived from the unchanged 49-patient baseline.
# Published batches may be attached later; they NEVER alter batch identities.
remaining = patient_registry.loc[~patient_registry["public_id"].isin(baseline_ids)].copy()
assert len(remaining) == 451

planned = []
for partition in ("validation", "train", "test"):
    part = remaining.loc[remaining["split"] == partition].copy()
    part["_numeric_id"] = part["public_id"].str.extract(r"(\d+)$")[0].astype(int)
    part = part.sort_values(["_numeric_id", "public_id"]).reset_index(drop=True)
    for j in range(0, len(part), BATCH_SIZE):
        k = 1 + j // BATCH_SIZE
        name = f"ribfrac-preprocessed-{partition}-{k:03d}"
        block = part.iloc[j:j + BATCH_SIZE].drop(columns="_numeric_id").copy()
        block["batch_name"] = name
        planned.append(block)

batch_plan = pd.concat(planned, ignore_index=True)
plan_cols = ["public_id", "split", "source", "batch_name"]
batch_plan = batch_plan[plan_cols]
assert len(batch_plan) == 451 and batch_plan["public_id"].nunique() == 451
assert set(batch_plan["public_id"]).isdisjoint(baseline_ids)
assert batch_plan["batch_name"].nunique() == 19
assert batch_plan.groupby("batch_name").size().max() <= 25

PLAN_PATH = PLAN_DIR / "batch_plan_500.csv"
if PLAN_PATH.is_file():
    previous = pd.read_csv(PLAN_PATH, dtype=str, keep_default_na=False)
    if not previous[plan_cols].equals(batch_plan):
        raise RuntimeError("Existing batch plan conflicts with recomputed plan. No overwrite performed.")
else:
    batch_plan.to_csv(PLAN_PATH, index=False)

batch_summary = batch_plan.groupby(["batch_name", "split"]).size().rename("patients")
print("Fixed plan stored at:", PLAN_PATH)
print(batch_summary.to_string())
print("New patients:", len(batch_plan), "| Planned datasets:", batch_plan["batch_name"].nunique())

## Cell 5 — Discover already attached published batches and reject duplicated patient IDs

In [ ]:
# IMPORTANT: Attach all previously published batch datasets in Kaggle -> Add Input.
# Only attached datasets can be counted as verified progress in this session.
attached_batches = {}
published_patient_owner = {}
for manifest_path in sorted(Path("/kaggle/input").rglob("metadata/batch_manifest.csv")):
    root = manifest_path.parent.parent
    data = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
    assert {"public_id", "split", "source", "batch_name"}.issubset(data.columns)
    assert data["batch_name"].nunique() == 1
    name = data["batch_name"].iloc[0]
    assert name in set(batch_plan["batch_name"]), f"Unrecognized dataset batch: {name}"
    if name in attached_batches:
        raise RuntimeError(f"Multiple attached copies/versions of {name}; detach duplicates.")
    expected = batch_plan.loc[batch_plan["batch_name"] == name, ["public_id", "split", "source", "batch_name"]]
    expected = expected.sort_values("public_id").reset_index(drop=True)
    actual = data[["public_id", "split", "source", "batch_name"]].sort_values("public_id").reset_index(drop=True)
    assert actual.equals(expected), f"Attached dataset manifest conflicts with authoritative plan: {name}"
    for pid in actual["public_id"]:
        if pid in published_patient_owner:
            raise RuntimeError(f"Patient {pid} duplicated across published batches")
        assert pid not in baseline_ids
        published_patient_owner[pid] = name
        img = root / "images" / f"{pid}-image.npy"
        lbl = root / "labels" / f"{pid}-label.npy"
        assert img.is_file() and lbl.is_file(), f"Missing CT/label pair in attached batch: {pid}"
    attached_batches[name] = root

print("Attached published batch datasets:", len(attached_batches))
for name, root in sorted(attached_batches.items()):
    print(" -", name, "at", root)
print("Baseline + attached-published patients:", len(baseline_ids) + len(published_patient_owner))
print("Still unaccounted for (could include unattached batches):", 500 - len(baseline_ids) - len(published_patient_owner))
print("Selected batch already attached:", SELECTED_BATCH in attached_batches)

## Cell 6 — Configure the selected staging folder and durable per-batch metadata

In [ ]:
assert SELECTED_BATCH in set(batch_plan["batch_name"]), f"Invalid batch name: {SELECTED_BATCH}"
current_manifest = batch_plan.loc[batch_plan["batch_name"] == SELECTED_BATCH].copy()
current_manifest = current_manifest.sort_values("public_id").reset_index(drop=True)
assert current_manifest["split"].nunique() == 1  # never mix train/val/test in a batch

BATCH_ROOT = BATCHES_DIR / SELECTED_BATCH
IMAGE_ROOT = BATCH_ROOT / "images"
LABEL_ROOT = BATCH_ROOT / "labels"
META_ROOT = BATCH_ROOT / "metadata"
STATUS_PATH = META_ROOT / "processing_status.csv"
MANIFEST_PATH = META_ROOT / "batch_manifest.csv"
CONFIG_PATH = META_ROOT / "preprocessing_config_500.json"
for folder in (IMAGE_ROOT, LABEL_ROOT, META_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

# Always keep stable source data assignments with each published dataset.
if MANIFEST_PATH.exists():
    old = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)
    assert old[["public_id", "split", "source", "batch_name"]].equals(current_manifest), (
        "Staging manifest differs from selected authoritative plan."
    )
else:
    current_manifest.to_csv(MANIFEST_PATH, index=False)

for src, dst in (
    (SPLIT_PATH, META_ROOT / "patient_split_500.csv"),
    (METADATA_PATH, META_ROOT / "combined_ribfrac_metadata.csv"),
    (PLAN_PATH, META_ROOT / "batch_plan_500.csv"),
):
    if dst.exists():
        assert hashlib.sha256(dst.read_bytes()).digest() == hashlib.sha256(src.read_bytes()).digest(), (
            f"Existing staged metadata differs from authoritative source: {dst}"
        )
    else:
        shutil.copy2(src, dst)

PREPROCESSING_CONFIG = {
    "target_spacing_mm": [1.0, 1.0, 1.0],
    "ct_interpolation": "linear",
    "label_interpolation": "nearest_neighbor",
    "hu_clip": [-1000.0, 400.0],
    "ct_normalization": "[0,1]",
    "ct_dtype": "float32",
    "label_dtype": "int16",
    "cohort_size": 500,
    "authoritative_split": EXPECTED_SPLITS,
    "selected_batch": SELECTED_BATCH,
    "batch_size_limit": BATCH_SIZE,
    "source_baseline_dataset": "waboke/ribfrac-preprocessed-final",
}
if CONFIG_PATH.is_file():
    assert json.loads(CONFIG_PATH.read_text()) == PREPROCESSING_CONFIG
else:
    CONFIG_PATH.write_text(json.dumps(PREPROCESSING_CONFIG, indent=2))

status_cols = ["public_id", "source", "split", "batch_name", "status", "started_at", "completed_at", "error", "image_size_bytes", "label_size_bytes", "ct_shape", "original_spacing"]
if STATUS_PATH.is_file():
    status_df = pd.read_csv(STATUS_PATH, dtype=str, keep_default_na=False)
    assert set(status_df["public_id"]) == set(current_manifest["public_id"])
    assert len(status_df) == len(current_manifest)
    for col in status_cols:
        assert col in status_df.columns, f"Corrupted staged processing status: {col}"
else:
    status_df = current_manifest.copy()
    status_df["status"] = "pending"
    for col in ["started_at", "completed_at", "error", "image_size_bytes", "label_size_bytes", "ct_shape", "original_spacing"]:
        status_df[col] = ""
    status_df = status_df[status_cols]
    status_df.to_csv(STATUS_PATH, index=False)

print("Selected batch:", SELECTED_BATCH)
print("Partition:", current_manifest["split"].unique().tolist())
print("Patients in this dataset:", len(current_manifest))
print("Staging root:", BATCH_ROOT)
print("Status:\n", status_df["status"].value_counts().to_string())

## Cell 7 — Retain the original validated 1-mm CT/label preprocessing

In [ ]:
TARGET_SPACING = np.array([1., 1., 1.], dtype=np.float32)
HU_MIN, HU_MAX = -1000.0, 400.0

def normalize_ct_hu(ct):
    ct = np.asarray(ct, dtype=np.float32)
    ct = np.clip(ct, HU_MIN, HU_MAX)
    return ((ct - HU_MIN) / (HU_MAX - HU_MIN)).astype(np.float32, copy=False)

def resample_volume(volume, original_spacing, target_spacing, order):
    factors = np.asarray(original_spacing, dtype=np.float32) / np.asarray(target_spacing, dtype=np.float32)
    return zoom(volume, zoom=factors, order=order, mode="nearest", prefilter=(order != 0))

def preprocess_nifti_pair(ct_nii, label_nii):
    ct = ct_nii.get_fdata(dtype=np.float32)
    label = label_nii.get_fdata(dtype=np.float32)
    ct_spacing = np.asarray(ct_nii.header.get_zooms()[:3], dtype=np.float32)
    label_spacing = np.asarray(label_nii.header.get_zooms()[:3], dtype=np.float32)
    if ct.ndim != 3 or label.ndim != 3 or ct.shape != label.shape:
        raise ValueError(f"CT/label 3D shape mismatch: {ct.shape} vs {label.shape}")
    if not np.allclose(ct_spacing, label_spacing, atol=1e-4):
        raise ValueError(f"CT/label spacing mismatch: {ct_spacing} vs {label_spacing}")
    # Stronger spatial integrity safeguard; does not alter the resampling/normalization.
    if not np.allclose(ct_nii.affine, label_nii.affine, rtol=1e-4, atol=1e-3):
        raise ValueError("CT and label NIfTI affine matrices differ; check spatial alignment")
    if not np.isfinite(ct).all() or not np.isfinite(label).all():
        raise ValueError("Non-finite CT or annotation values")
    if not np.allclose(label, np.rint(label)):
        raise ValueError("Non-integer annotation values")
    if np.nanmin(label) < np.iinfo(np.int16).min or np.nanmax(label) > np.iinfo(np.int16).max:
        raise ValueError("Annotation ID exceeds int16 range")
    ct = normalize_ct_hu(ct)
    ct_out = resample_volume(ct, ct_spacing, TARGET_SPACING, order=1).astype(np.float32)
    label_out = np.rint(resample_volume(label, label_spacing, TARGET_SPACING, order=0)).astype(np.int16)
    if ct_out.shape != label_out.shape or not np.isfinite(ct_out).all():
        raise ValueError("Invalid resampled CT/label pair")
    if ct_out.min() < -1e-5 or ct_out.max() > 1 + 1e-5:
        raise ValueError("Normalized CT outside [0,1]")
    return {"ct": ct_out, "label": label_out, "original_spacing": ct_spacing.tolist(), "target_spacing": TARGET_SPACING.tolist()}

print("Original HU clipping and CT/label interpolation retained.")

## Cell 8 — Zenodo ZIP range-download helpers (on-demand, no full CT ZIP download)

In [ ]:
# This code uses the original Notebook 02 ZIP64 indexing strategy.
# Unlike the original loader, it streams individual NIfTI members to temporary files.
ZIP_INDEX_CACHE = {}
HTTP_SESSION = requests.Session()
HTTP_SESSION.headers.update({"User-Agent": "Mozilla/5.0", "Accept-Encoding": "identity"})

def http_get_range(url, start, end, timeout=180, attempts=4):
    if start < 0 or end < start:
        raise ValueError("Invalid HTTP byte range")
    headers = {"Range": f"bytes={start}-{end}"}
    for attempt in range(attempts):
        try:
            with HTTP_SESSION.get(url, headers=headers, stream=True, timeout=(30, timeout), allow_redirects=True) as r:
                r.raise_for_status()
                if r.status_code != 206:
                    raise RuntimeError(f"Server must return HTTP 206; got {r.status_code} for {url}")
                cr = r.headers.get("Content-Range", "")
                if not cr.startswith(f"bytes {start}-{end}/"):
                    raise RuntimeError(f"Unexpected HTTP Content-Range: {cr}")
                result = b"".join(r.iter_content(chunk_size=1024*1024))
                if len(result) != end-start+1:
                    raise RuntimeError(f"Incomplete HTTP range: got {len(result)} bytes")
                return result
        except (requests.RequestException, RuntimeError) as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            if (status is not None and status in (401, 403, 404)) or (isinstance(e, RuntimeError) and "Server must return" in str(e)):
                raise RuntimeError(f"Zenodo HTTP range unavailable: {e}. Attach raw NIfTI files if needed.") from e
            if attempt == attempts - 1:
                raise
            time.sleep(min(15, 2**attempt))

def get_remote_size(url):
    try:
        r = HTTP_SESSION.head(url, timeout=60, allow_redirects=True)
        if r.ok and int(r.headers.get("Content-Length", "0")) > 0:
            return int(r.headers["Content-Length"])
    except (requests.RequestException, ValueError):
        pass
    # A single-byte GET is a safe fallback, not a full archive download.
    headers = {"Range": "bytes=0-0"}
    with HTTP_SESSION.get(url, headers=headers, stream=True, timeout=60, allow_redirects=True) as r:
        r.raise_for_status()
        if r.status_code != 206:
            raise RuntimeError(f"Source ZIP does not support required byte ranges: HTTP {r.status_code}")
        match = re.fullmatch(r"bytes\s+0-0/(\d+)", r.headers.get("Content-Range", "").strip())
        if not match:
            raise RuntimeError("Missing/invalid Content-Range for archive size")
        return int(match.group(1))

def read_zip64_central_directory(url):
    size = get_remote_size(url)
    tail_size = min(size, 1024 * 1024)
    tail = http_get_range(url, size-tail_size, size-1)
    eocd_pos = tail.rfind(b"PK\x05\x06")
    if eocd_pos < 0:
        raise RuntimeError("ZIP end-of-central-directory record not found")
    eocd = tail[eocd_pos:eocd_pos+22]
    if len(eocd) < 22:
        raise RuntimeError("Truncated ZIP EOCD")
    _, _, _, _, count, cd_size, cd_offset, _ = struct.unpack("<4sHHHHIIH", eocd)
    if cd_size == 0xffffffff or cd_offset == 0xffffffff:
        locator_pos = tail.rfind(b"PK\x06\x07", 0, eocd_pos)
        if locator_pos < 0:
            raise RuntimeError("ZIP64 locator not found")
        _, _, z64_offset, _ = struct.unpack("<4sIQI", tail[locator_pos:locator_pos+20])
        z64 = http_get_range(url, z64_offset, z64_offset+55)
        if z64[:4] != b"PK\x06\x06":
            raise RuntimeError("ZIP64 EOCD record not found")
        count = struct.unpack_from("<Q", z64, 32)[0]
        cd_size = struct.unpack_from("<Q", z64, 40)[0]
        cd_offset = struct.unpack_from("<Q", z64, 48)[0]
    if cd_size > 64*1024*1024:
        raise RuntimeError("Unexpectedly large ZIP central directory")
    data = http_get_range(url, cd_offset, cd_offset+cd_size-1)
    pos, entries = 0, {}
    for _ in range(count):
        if data[pos:pos+4] != b"PK\x01\x02":
            raise RuntimeError("Invalid ZIP central-directory signature")
        fields = struct.unpack("<4s6H3I5H2I", data[pos:pos+46])
        (sig, ver_made, ver_need, flags, compression, mtime, mdate, crc,
         compressed_size, uncompressed_size, fnlen, extralen, commentlen,
         disk_start, intattr, extattr, offset) = fields
        start = pos+46
        filename = data[start:start+fnlen].decode("utf-8", errors="replace")
        extra = data[start+fnlen:start+fnlen+extralen]
        if (compressed_size == 0xffffffff or uncompressed_size == 0xffffffff or offset == 0xffffffff):
            off = 0
            z64 = None
            j = 0
            while j+4 <= len(extra):
                eid, elen = struct.unpack("<HH", extra[j:j+4])
                if eid == 1:
                    z64 = extra[j+4:j+4+elen]
                    break
                j += 4+elen
            if z64 is None:
                raise RuntimeError(f"Missing ZIP64 sizes for {filename}")
            if uncompressed_size == 0xffffffff:
                uncompressed_size = struct.unpack_from("<Q", z64, off)[0]; off += 8
            if compressed_size == 0xffffffff:
                compressed_size = struct.unpack_from("<Q", z64, off)[0]; off += 8
            if offset == 0xffffffff:
                offset = struct.unpack_from("<Q", z64, off)[0]; off += 8
        entries[filename] = {
            "compression": int(compression), "compressed_size": int(compressed_size),
            "uncompressed_size": int(uncompressed_size), "local_header_offset": int(offset),
            "crc32": int(crc)
        }
        pos = start + fnlen+extralen+commentlen
    return entries

def find_patient_member(entries, patient_id, kind):
    exact = f"{patient_id}-{kind}.nii.gz".lower()
    candidates = [name for name in entries if Path(name).name.lower() == exact]
    if len(candidates) == 1:
        return candidates[0]
    # Guard against RibFrac10 mistakenly matching RibFrac100.
    regex = re.compile(rf"^{re.escape(patient_id)}(?!\d).*{re.escape(kind)}.*\.nii(?:\.gz)?$", re.I)
    candidates = [name for name in entries if regex.match(Path(name).name)]
    if len(candidates) != 1:
        raise FileNotFoundError(f"Expected exactly one {kind} NIfTI for {patient_id}; found {candidates}")
    return candidates[0]

def extract_zip_member(url, entry, destination, chunk_size=4*1024*1024):
    destination = Path(destination)
    off = entry["local_header_offset"]
    header = http_get_range(url, off, off+29)
    if header[:4] != b"PK\x03\x04":
        raise RuntimeError("Invalid ZIP local header")
    parsed = struct.unpack("<4s5H3I2H", header)
    local_method, fnlen, extralen = parsed[3], parsed[-2], parsed[-1]
    if local_method != entry["compression"]:
        raise RuntimeError("Local vs central ZIP compression mismatch")
    start = off + 30 + fnlen + extralen
    total = entry["compressed_size"]
    method = entry["compression"]
    if method not in (0, 8):
        raise RuntimeError(f"Unsupported ZIP compression: {method}")
    decoder = zlib.decompressobj(-15) if method == 8 else None
    written, crc = 0, 0
    with destination.open("wb") as f:
        for cursor in range(0, total, chunk_size):
            count = min(chunk_size, total - cursor)
            payload = http_get_range(url, start+cursor, start+cursor+count-1)
            decoded = decoder.decompress(payload) if decoder is not None else payload
            f.write(decoded)
            written += len(decoded)
            crc = zlib.crc32(decoded, crc)
        if decoder is not None:
            final = decoder.flush()
            f.write(final)
            written += len(final)
            crc = zlib.crc32(final, crc)
    if written != entry["uncompressed_size"] or (crc & 0xffffffff) != entry["crc32"]:
        destination.unlink(missing_ok=True)
        raise RuntimeError("Downloaded ZIP member failed size or CRC check")
    return destination

def get_source_index(source_name, kind):
    key = (source_name, kind)
    if key not in ZIP_INDEX_CACHE:
        url = SOURCES[source_name][f"{kind}_url"]
        print("Indexing remote", kind, "ZIP for", source_name)
        ZIP_INDEX_CACHE[key] = read_zip64_central_directory(url)
    return ZIP_INDEX_CACHE[key]

print("On-demand ZIP64 helpers ready. No remote data downloaded by this cell.")

## Cell 9 — Locate optional attached raw NIfTI files or stream individual Zenodo members

In [ ]:
# Local raw data are optional; otherwise load per-patient NIfTI from the original ZIPs.
LOCAL_RAW_INDEX = {}
for input_dir, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if not filename.lower().endswith((".nii.gz", ".nii")):
            continue
        match = re.fullmatch(r"(RibFrac\d+)-(image|label)\.nii(?:\.gz)?", filename, flags=re.I)
        if match:
            key = (match.group(1), match.group(2).lower())
            LOCAL_RAW_INDEX.setdefault(key, []).append(Path(input_dir) / filename)

for key, paths in LOCAL_RAW_INDEX.items():
    if len(paths) > 1:
        raise RuntimeError(f"Multiple raw files for {key}: {paths}")


def load_patient_nifti_from_source(patient_id, source_name, temporary_dir):
    img_local = LOCAL_RAW_INDEX.get((patient_id, "image"))
    lbl_local = LOCAL_RAW_INDEX.get((patient_id, "label"))
    if img_local and lbl_local:
        return nib.load(str(img_local[0])), nib.load(str(lbl_local[0]))
    if bool(img_local) != bool(lbl_local):
        raise RuntimeError(f"Only one local raw NIfTI is available for {patient_id}; attach its matching pair")
    temp = Path(temporary_dir)
    nii_paths = {}
    for kind in ("image", "label"):
        index = get_source_index(source_name, kind)
        member = find_patient_member(index, patient_id, kind)
        ext = ".nii.gz" if member.lower().endswith(".nii.gz") else ".nii"
        output_path = temp / f"{patient_id}-{kind}{ext}"
        extract_zip_member(SOURCES[source_name][f"{kind}_url"], index[member], output_path)
        nii_paths[kind] = output_path
    return nib.load(str(nii_paths["image"])), nib.load(str(nii_paths["label"]))

print("Matching local raw NIfTI files indexed:", len(LOCAL_RAW_INDEX))
print("Source fallback: streaming individual ZIP members (network required).")

## Cell 10 — Preflight, file safety, status updates, atomic array saving

In [ ]:
print("="*70)
print("PREPROCESSING PREFLIGHT —", SELECTED_BATCH)
print("="*70)
print("Patient IDs:", current_manifest["public_id"].tolist())
print("Partition:", current_manifest["split"].unique().tolist())
print("Already attached/published:", SELECTED_BATCH in attached_batches)
print(f"Available working disk: {shutil.disk_usage('/kaggle/working').free/GiB:.2f} GiB")
print("Staging directory:", BATCH_ROOT)


def validate_pair(patient_id, image_path, label_path):
    image_path, label_path = Path(image_path), Path(label_path)
    if not image_path.is_file() or not label_path.is_file():
        return False
    try:
        x = np.load(image_path, mmap_mode="r", allow_pickle=False)
        y = np.load(label_path, mmap_mode="r", allow_pickle=False)
        good = x.ndim == y.ndim == 3 and x.shape == y.shape and x.dtype == np.float32 and y.dtype == np.int16
        good = bool(good and image_path.stat().st_size > 128 and label_path.stat().st_size > 128)
        del x, y
        return good
    except (OSError, ValueError, TypeError):
        return False


def update_batch_status(patient_id, **fields):
    df = pd.read_csv(STATUS_PATH, dtype=str, keep_default_na=False)
    ids = df.index[df["public_id"] == patient_id]
    if len(ids) != 1:
        raise RuntimeError(f"Missing or duplicate status row for {patient_id}")
    for k, v in fields.items():
        if k not in df.columns:
            raise KeyError(k)
        df.loc[ids[0], k] = str(v)
    tmp = STATUS_PATH.with_suffix(".tmp.csv")
    df.to_csv(tmp, index=False)
    os.replace(tmp, STATUS_PATH)


def save_patient_pair(patient_id, x, y):
    img = IMAGE_ROOT / f"{patient_id}-image.npy"
    lbl = LABEL_ROOT / f"{patient_id}-label.npy"
    if img.exists() or lbl.exists():
        if validate_pair(patient_id, img, lbl):
            raise FileExistsError(f"Valid patient files already exist for {patient_id}")
        # A crash may have left a partial pair; do not silently overwrite it.
        raise RuntimeError(f"Incomplete files already exist for {patient_id}; inspect manually: {img}, {lbl}")
    img_tmp = IMAGE_ROOT / f".{patient_id}-image.tmp.npy"
    lbl_tmp = LABEL_ROOT / f".{patient_id}-label.tmp.npy"
    try:
        with img_tmp.open("wb") as f:
            np.save(f, x.astype(np.float32, copy=False), allow_pickle=False)
        with lbl_tmp.open("wb") as f:
            np.save(f, y.astype(np.int16, copy=False), allow_pickle=False)
        os.replace(img_tmp, img)
        os.replace(lbl_tmp, lbl)
        if not validate_pair(patient_id, img, lbl):
            raise RuntimeError(f"Newly saved patient files failed verification: {patient_id}")
    finally:
        img_tmp.unlink(missing_ok=True)
        lbl_tmp.unlink(missing_ok=True)
    return img, lbl

print("Staged status:\n", pd.read_csv(STATUS_PATH)["status"].value_counts().to_string())
print("No CT processing executed by this cell.")

## Cell 11 — Process up to 25 patients in the selected batch (explicit opt-in; may stop for storage safety)

In [ ]:
# IMPORTANT: Enable RUN_PROCESSING=True in Cell 2 and rerun that cell,
# then rerun this cell. This attempts up to 25 patients; storage or download errors can stop it early.

def process_next_patients(max_patients=PATIENTS_PER_RUN):
    if SELECTED_BATCH in attached_batches:
        raise RuntimeError("Selected batch is already published and attached; do not preprocess it again.")
    registry = current_manifest.set_index("public_id")
    current_status = pd.read_csv(STATUS_PATH, dtype=str, keep_default_na=False).set_index("public_id")
    completed, failed, skipped = [], [], []
    attempts = 0
    for pid, row in registry.iterrows():
        img = IMAGE_ROOT / f"{pid}-image.npy"
        lbl = LABEL_ROOT / f"{pid}-label.npy"
        if validate_pair(pid, img, lbl):
            skipped.append(pid)
            if current_status.loc[pid, "status"] != "completed":
                update_batch_status(pid, status="completed", image_size_bytes=img.stat().st_size, label_size_bytes=lbl.stat().st_size, error="")
            continue
        if img.exists() or lbl.exists():
            raise RuntimeError(f"Partial output for {pid}; inspect manually before continuing: {img}, {lbl}")
        if current_status.loc[pid, "status"] == "failed" and not RETRY_FAILED:
            continue
        if attempts >= max_patients:
            break
        free_gib = shutil.disk_usage("/kaggle/working").free / GiB
        if free_gib < MIN_FREE_GIB:
            print(f"SAFETY STOP: only {free_gib:.2f} GiB free (< {MIN_FREE_GIB:.2f} GiB).")
            break
        attempts += 1
        source_name = row["source"]
        update_batch_status(pid, status="processing", started_at=time.strftime("%Y-%m-%d %H:%M:%S"), error="")
        print(f"\n[{attempts}/{max_patients}] {pid} ({row['split']}, {source_name}); free={free_gib:.2f} GiB")
        try:
            with tempfile.TemporaryDirectory(dir=TEMP_DIR) as temp:
                ct_nii, label_nii = load_patient_nifti_from_source(pid, source_name, temp)
                result = preprocess_nifti_pair(ct_nii, label_nii)
                del ct_nii, label_nii
                x, y = result["ct"], result["label"]
                if not np.isfinite(x).all() or x.shape != y.shape:
                    raise ValueError("Invalid processed volume")
                img, lbl = save_patient_pair(pid, x, y)
                update_batch_status(
                    pid, status="completed", completed_at=time.strftime("%Y-%m-%d %H:%M:%S"),
                    image_size_bytes=img.stat().st_size, label_size_bytes=lbl.stat().st_size,
                    ct_shape=list(x.shape), original_spacing=result["original_spacing"], error=""
                )
                print(f"COMPLETED: {pid}, shape={x.shape}, CT+label={(img.stat().st_size+lbl.stat().st_size)/GiB:.3f} GiB")
                completed.append(pid)
                del x, y, result
        except Exception as exc:
            update_batch_status(pid, status="failed", error=repr(exc))
            print(f"FAILED: {pid}: {exc}")
            failed.append(pid)
            if any(code in str(exc).lower() for code in ("403", "429", "http range", "rate limit", "no space")):
                print("Stopping after a remote access/storage failure; fix the source problem first.")
                break
        finally:
            gc.collect()
    print("\nRun summary — completed:", completed, "failed:", failed, "already present:", len(skipped))
    current = pd.read_csv(STATUS_PATH, dtype=str, keep_default_na=False)
    print("Current batch statuses:\n", current["status"].value_counts().to_string())
    return current

if RUN_PROCESSING:
    print("Attempting up to 25 patients in this session; storage guard may stop earlier.")
    status_df = process_next_patients()
else:
    print("SAFE PREVIEW: RUN_PROCESSING=False. No CT images downloaded or preprocessed.")

## Cell 12 — Validate complete batch, create SHA-256 manifest (before publishing)

In [ ]:
# This cell runs in read-only mode until all assigned patient arrays exist.

def file_sha256(path, block_size=4*1024*1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(block_size), b""):
            digest.update(part)
    return digest.hexdigest()


def verify_current_batch(write_integrity=True):
    records, missing = [], []
    allowed = set(current_manifest["public_id"])
    actual_img = {p.name.removesuffix("-image.npy") for p in IMAGE_ROOT.glob("*-image.npy")}
    actual_lbl = {p.name.removesuffix("-label.npy") for p in LABEL_ROOT.glob("*-label.npy")}
    assert actual_img.issubset(allowed) and actual_lbl.issubset(allowed), (
        "Staging folder contains an unplanned patient; publication is blocked."
    )
    for pid in current_manifest["public_id"]:
        img = IMAGE_ROOT / f"{pid}-image.npy"
        lbl = LABEL_ROOT / f"{pid}-label.npy"
        if not validate_pair(pid, img, lbl):
            missing.append(pid)
        else:
            records.extend([
                {"public_id": pid, "file": f"images/{img.name}", "size_bytes": img.stat().st_size},
                {"public_id": pid, "file": f"labels/{lbl.name}", "size_bytes": lbl.stat().st_size},
            ])
    done = not missing and len(actual_img) == len(allowed) and len(actual_lbl) == len(allowed)
    if done:
        status = pd.read_csv(STATUS_PATH, dtype=str, keep_default_na=False)
        assert set(status["public_id"]) == allowed and (status["status"] == "completed").all(), (
            "All arrays exist, but batch processing statuses are inconsistent."
        )
        if write_integrity:
            for r in records:
                r["sha256"] = file_sha256(BATCH_ROOT / r["file"])
            manifest = pd.DataFrame(records)
            temp = META_ROOT / "file_integrity.tmp.csv"
            manifest.to_csv(temp, index=False)
            os.replace(temp, META_ROOT / "file_integrity.csv")
            summary = {"batch": SELECTED_BATCH, "patients": len(allowed), "file_count": len(records), "split": current_manifest["split"].iloc[0], "status": "fully_verified"}
            (META_ROOT / "validation_report.json").write_text(json.dumps(summary, indent=2))
    print(f"Verified complete pairs: {len(allowed)-len(missing)}/{len(allowed)}")
    print("Missing/incomplete:", missing)
    print("READY TO PUBLISH:", done)
    if done and write_integrity:
        print("SHA-256 file integrity inventory saved:", META_ROOT / "file_integrity.csv")
    return done

BATCH_READY_TO_PUBLISH = verify_current_batch(write_integrity=True)

## Cell 13 — Publish exactly one completed batch (disabled until explicitly authorized)

In [ ]:
# To upload, set PUBLISH_BATCH=True and PUBLISH_CONFIRMATION=SELECTED_BATCH
# in Cell 2, rerun Cell 2, then rerun this cell.
DATASET_HANDLE = f"{DATASET_OWNER}/{SELECTED_BATCH}"
print("Proposed Kaggle dataset:", DATASET_HANDLE)

if PUBLISH_BATCH:
    assert PUBLISH_CONFIRMATION == SELECTED_BATCH, (
        "Publishing blocked: enter the exact selected batch name in PUBLISH_CONFIRMATION."
    )
    assert SELECTED_BATCH not in attached_batches, (
        "Publishing blocked: a dataset with this batch name is already attached."
    )
    assert verify_current_batch(write_integrity=True), "Cannot publish incomplete patient batch"
    assert len(current_manifest) <= BATCH_SIZE
    import kagglehub
    kagglehub.dataset_upload(
        DATASET_HANDLE,
        str(BATCH_ROOT),
        version_notes=(f"Verified RibFrac batch {SELECTED_BATCH}; "
                       f"{len(current_manifest)} pairs; 350/75/75 authoritative split preserved"),
        ignore_patterns=["*.tmp*", "*.partial", "cache/", "_tmp/"],
    )
    print("UPLOAD RETURNED. This is not independent verification.")
    print("Check Kaggle dataset Versions; attach the NEW dataset version to this notebook.")
else:
    print("SAFE PREVIEW: PUBLISH_BATCH=False. No Kaggle upload performed.")

## Cell 14 — Reattach and independently verify the published batch; optional safe cleanup

In [ ]:
# Run this cell AFTER attaching the published batch as a Kaggle input.
# To delete the local staging folder, set DELETE_STAGING_AFTER_VERIFICATION=True
# and DELETE_CONFIRMATION=SELECTED_BATCH in Cell 2, then rerun this cell.

def verify_attached_published_batch():
    matching = []
    for path in Path("/kaggle/input").rglob("metadata/batch_manifest.csv"):
        data = pd.read_csv(path, dtype=str, keep_default_na=False)
        if "batch_name" in data.columns and len(data) and data["batch_name"].nunique() == 1 and data["batch_name"].iat[0] == SELECTED_BATCH:
            matching.append(path.parent.parent)
    if len(matching) != 1:
        print(f"Cannot verify: found {len(matching)} attached copies of {SELECTED_BATCH} (need exactly one).")
        return False
    remote_root = matching[0]
    expected = current_manifest[["public_id", "source", "split", "batch_name"]].sort_values("public_id").reset_index(drop=True)
    remote_manifest = pd.read_csv(remote_root / "metadata/batch_manifest.csv", dtype=str, keep_default_na=False)
    actual = remote_manifest[["public_id", "source", "split", "batch_name"]].sort_values("public_id").reset_index(drop=True)
    if not expected.equals(actual):
        raise RuntimeError("Published manifest does not match the selected authoritative batch")
    remote_integrity = remote_root / "metadata/file_integrity.csv"
    if not remote_integrity.is_file():
        raise RuntimeError("Published dataset lacks the SHA-256 integrity inventory")
    integrity = pd.read_csv(remote_integrity, dtype=str, keep_default_na=False)
    assert len(integrity) == 2*len(current_manifest)
    expected_paths = ({f"images/{p}-image.npy" for p in current_manifest["public_id"]}
                      | {f"labels/{p}-label.npy" for p in current_manifest["public_id"]})
    assert set(integrity["file"]) == expected_paths and integrity["file"].nunique() == len(expected_paths)
    for _, row in integrity.iterrows():
        p = remote_root / row["file"]
        if not p.is_file() or p.stat().st_size != int(row["size_bytes"]):
            raise RuntimeError(f"Missing or incorrect published file: {p}")
        if file_sha256(p) != row["sha256"]:
            raise RuntimeError(f"SHA-256 mismatch for published file: {p}")
    print("PUBLISHED BATCH VERIFIED:", remote_root)
    print("Patients:", len(current_manifest), "| Files:", len(integrity), "| SHA-256: PASS")
    return True

PERSISTED_COPY_VERIFIED = verify_attached_published_batch()
if DELETE_STAGING_AFTER_VERIFICATION:
    assert DELETE_CONFIRMATION == SELECTED_BATCH, "Explicit deletion confirmation missing"
    assert PERSISTED_COPY_VERIFIED, "No verified published copy; deletion blocked"
    assert BATCH_ROOT.parent.resolve() == BATCHES_DIR.resolve(), "Unsafe staging path"
    assert BATCH_ROOT.name == SELECTED_BATCH, "Staging folder identity mismatch"
    shutil.rmtree(BATCH_ROOT)
    print("LOCAL STAGING REMOVED after independent verification:", BATCH_ROOT)
else:
    print("Local staging retained; automatic deletion disabled.")

## Cell 15 — Cohort progress report and next-batch instructions

In [ ]:
# Count only the fixed baseline and verified, ATTACHED published datasets.
# If a previously uploaded dataset is missing from Kaggle inputs, attach it first.
verified_published_ids = set()
verified_splits = []
for path in Path("/kaggle/input").rglob("metadata/batch_manifest.csv"):
    root = path.parent.parent
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not len(df) or df["batch_name"].nunique() != 1:
        continue
    name = df["batch_name"].iat[0]
    if name not in set(batch_plan["batch_name"]):
        continue
    expected = batch_plan.loc[batch_plan["batch_name"] == name]
    assert set(df["public_id"]) == set(expected["public_id"])
    for pid in df["public_id"]:
        assert pid not in baseline_ids and pid not in verified_published_ids
        assert validate_pair(pid, root / "images" / f"{pid}-image.npy", root / "labels" / f"{pid}-label.npy")
        verified_published_ids.add(pid)

counted_ids = baseline_ids | verified_published_ids
counts = patient_registry.loc[patient_registry["public_id"].isin(counted_ids), "split"].value_counts()
print("="*65)
print("RIBFRAC — PERSISTENT PATIENT COVERAGE")
print("="*65)
print("Baseline patient pairs:", len(baseline_ids))
print("Patients in attached, complete batch datasets:", len(verified_published_ids))
print("TOTAL PERSISTED & ATTACHED:", len(counted_ids), "/ 500")
print("Not yet accounted for by attached datasets:", 500 - len(counted_ids))
for partition in ("train", "validation", "test"):
    print(f"  {partition:10s}: {counts.get(partition, 0):3d} / {EXPECTED_SPLITS[partition]}")
print("\nAttach all prior published batch datasets to obtain a complete progress count.")
print("To continue: change SELECTED_BATCH in Cell 2, rerun setup Cells 2–10, then process the next batch.")
print("Never use the original 'test' patients for training or model selection.")